# مسئلهٔ ۱ — V2 / مرحلهٔ ۱: Data Audit و manifest مرکزی

این نوت‌بوک فقط داده‌های انتخاب‌شدهٔ ۶۰۰ ویدئویی را بررسی می‌کند و هیچ فایل MP4 یا خروجی V1 را تغییر/حذف نمی‌کند. خروجی اصلی آن `P:\NexarCollisionData\video_manifest_v2.csv` است؛ از این فایل در همهٔ مراحل بعدی V2 استفاده می‌کنیم.

برچسب فقط از وجود `time_of_event` ساخته می‌شود. مسیر فایل، شناسهٔ ویدئو و زمان رویداد هرگز ورودی مدل نخواهند بود.

In [12]:
from __future__ import annotations

from hashlib import sha256
from pathlib import Path
import json
import math

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

DATA_ROOT = Path(r'P:\NexarCollisionData')
SELECTED_PATH = DATA_ROOT / 'selected_train_600.csv'
SPLITS_PATH = DATA_ROOT / 'video_splits.csv'
MANIFEST_PATH = DATA_ROOT / 'video_manifest_v2.csv'
EXACT_DUPLICATES_PATH = DATA_ROOT / 'exact_duplicate_groups_v2.csv'
NEAR_DUPLICATES_PATH = DATA_ROOT / 'near_duplicate_candidates_v2.csv'
SUMMARY_PATH = DATA_ROOT / 'data_audit_summary_v2.json'

# فقط برای audit: پنج نقطهٔ یکنواخت از هر ویدئو را می‌خوانیم.
SAMPLE_RATIOS = (0.10, 0.30, 0.50, 0.70, 0.90)
BLACK_MEAN_THRESHOLD = 5.0
# آستانه فقط برای «نامزد بررسی دستی» است؛ حذف یا جابه‌جایی split خودکار نداریم.
NEAR_DUPLICATE_HAMMING_THRESHOLD = 6.0

assert SELECTED_PATH.exists(), f'Missing: {SELECTED_PATH}'
assert SPLITS_PATH.exists(), f'Missing: {SPLITS_PATH}'
print(f'Data root: {DATA_ROOT}')

Data root: P:\NexarCollisionData


In [13]:
def canonical_video_id(value: object) -> str:
    """Make IDs joinable while keeping the real filename separately in video_path."""
    text = str(value).strip()
    try:
        return str(int(text))
    except ValueError:
        return text

selected = pd.read_csv(SELECTED_PATH)
splits = pd.read_csv(SPLITS_PATH)

required_selected = {
    'local_path', 'time_of_event', 'light_conditions', 'weather', 'scene'
}
required_splits = {'video_id', 'video_path', 'label', 'split'}
assert required_selected.issubset(selected.columns), selected.columns.tolist()
assert required_splits.issubset(splits.columns), splits.columns.tolist()

selected = selected.copy()
selected['video_path'] = selected['local_path'].astype(str)
selected['video_id'] = selected['video_path'].map(lambda p: canonical_video_id(Path(p).stem))
selected['time_of_event'] = pd.to_numeric(selected['time_of_event'], errors='coerce')
# تعریف رسمی برچسب V2؛ ستون label قبلی فقط برای کنترل سازگاری نگه داشته می‌شود.
selected['label_from_event'] = selected['time_of_event'].notna().astype(int)

splits = splits.copy()
splits['video_id'] = splits['video_id'].map(canonical_video_id)

assert len(selected) == 600, f'Expected 600 selected videos, got {len(selected)}'
assert selected['video_id'].is_unique, 'Duplicate video_id in selected_train_600.csv'
assert splits['video_id'].is_unique, 'Duplicate video_id in video_splits.csv'
assert set(selected['video_id']) == set(splits['video_id']), 'Selected videos and frozen V1 split do not match'
if 'label' in selected.columns:
    assert (selected['label'].astype(int) == selected['label_from_event']).all(), 'Existing label disagrees with time_of_event'

source_columns = [
    'video_id', 'video_path', 'label_from_event', 'time_of_event',
    'weather', 'light_conditions', 'scene'
]
manifest_source = selected[source_columns].rename(columns={'label_from_event': 'label'})
manifest_source = manifest_source.merge(
    splits[['video_id', 'split']], on='video_id', how='left', validate='one_to_one'
)
assert manifest_source['split'].notna().all(), 'At least one video has no frozen split'

print('Frozen split and class counts:')
display(pd.crosstab(manifest_source['split'], manifest_source['label']))
display(manifest_source.head(3))

Frozen split and class counts:


label,0,1
split,,
train,240,240
validation,60,60


,video_id,video_path,label,time_of_event,weather,light_conditions,scene,split
0,1094,P:\NexarCollisionData\train\negative\01094.mp4,0,NaN,Clear,Twilight,Highway,train
1,343,P:\NexarCollisionData\train\positive\00343.mp4,1,18.826,Cloudy,Normal,Urban,train
2,444,P:\NexarCollisionData\train\positive\00444.mp4,1,19.250,Cloudy,Normal,Urban,train


In [14]:
def file_sha256(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def perceptual_hash_hex(frame_bgr: np.ndarray) -> str:
    """Standard-style 64-bit DCT pHash, stronger than a global average hash."""
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)
    low_frequency = cv2.dct(small)[:8, :8]
    median = np.median(low_frequency.reshape(-1)[1:])  # DC component is excluded.
    bits = (low_frequency >= median).reshape(-1).astype(np.uint8)
    return np.packbits(bits).tobytes().hex()

def inspect_video(row: pd.Series) -> dict:
    """Inspect one MP4 and return audit values; it never deletes or modifies the file."""
    path = Path(row.video_path)
    errors: list[str] = []
    result = {
        'file_exists': path.is_file(),
        'file_size_bytes': path.stat().st_size if path.is_file() else np.nan,
        'sha256': None,
        'fps': np.nan,
        'frame_count': np.nan,
        'duration': np.nan,
        'width': np.nan,
        'height': np.nan,
        'first_frame_readable': False,
        'sample_mean_intensity': np.nan,
        'sample_min_intensity': np.nan,
        'all_sampled_frames_black': False,
        'phash_0': None,
        'phash_1': None,
        'phash_2': None,
    }

    if not path.is_file():
        errors.append('missing_file')
        result['is_valid'] = False
        result['error_reason'] = ';'.join(errors)
        return result

    try:
        result['sha256'] = file_sha256(path)
    except OSError as exc:
        errors.append(f'hash_error:{type(exc).__name__}')

    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        errors.append('cannot_open')
        cap.release()
        result['is_valid'] = False
        result['error_reason'] = ';'.join(errors)
        return result

    try:
        fps = float(cap.get(cv2.CAP_PROP_FPS))
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = (frame_count / fps) if fps > 0 else np.nan
        result.update({
            'fps': fps, 'frame_count': frame_count, 'duration': duration,
            'width': width, 'height': height,
        })

        if not np.isfinite(fps) or fps <= 0:
            errors.append('invalid_fps')
        if frame_count <= 0:
            errors.append('no_frames')
        if width <= 0 or height <= 0:
            errors.append('invalid_resolution')
        if not np.isfinite(duration) or duration <= 0:
            errors.append('invalid_duration')

        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        first_ok, _ = cap.read()
        result['first_frame_readable'] = bool(first_ok)
        if not first_ok:
            errors.append('cannot_read_first_frame')

        sample_means = []
        if np.isfinite(duration) and duration > 0:
            for sample_idx, ratio in enumerate(SAMPLE_RATIOS):
                timestamp_s = min(duration * ratio, max(duration - 1.0 / max(fps, 1.0), 0.0))
                cap.set(cv2.CAP_PROP_POS_MSEC, timestamp_s * 1000.0)
                ok, frame = cap.read()
                if not ok or frame is None:
                    errors.append(f'cannot_read_sample_{sample_idx}')
                    continue
                sample_means.append(float(frame.mean()))
                result[f'phash_{sample_idx}'] = perceptual_hash_hex(frame)

        if sample_means:
            result['sample_mean_intensity'] = float(np.mean(sample_means))
            result['sample_min_intensity'] = float(np.min(sample_means))
            result['all_sampled_frames_black'] = bool(max(sample_means) < BLACK_MEAN_THRESHOLD)
            if result['all_sampled_frames_black']:
                errors.append('all_sampled_frames_black')
        else:
            errors.append('no_readable_samples')

        event_time = row.time_of_event
        if int(row.label) == 1:
            if pd.isna(event_time):
                errors.append('positive_without_event_time')
            elif not np.isfinite(duration) or not (0.0 <= float(event_time) <= duration):
                errors.append('event_time_out_of_range')
    except Exception as exc:
        # A broken codec or a truncated stream must be reported, never stop the audit silently.
        errors.append(f'inspection_error:{type(exc).__name__}')
    finally:
        cap.release()

    result['is_valid'] = len(errors) == 0
    result['error_reason'] = None if result['is_valid'] else ';'.join(dict.fromkeys(errors))
    return result


## اجرای audit

این سلول برای هر MP4 یک بار فایل را باز می‌کند و SHA-256 کامل آن را محاسبه می‌کند؛ روی ۶۰۰ ویدئو ممکن است چند دقیقه زمان ببرد. اجرای دوباره فقط گزارش V2 را بازنویسی می‌کند و دانلود یا حذف انجام نمی‌دهد.

In [15]:
audit_records = []
audit_records = []
for _, row in tqdm(manifest_source.iterrows(), total=len(manifest_source), desc='Auditing MP4 files'):
    audit_records.append(inspect_video(row))

audit = pd.DataFrame(audit_records)
manifest = pd.concat([manifest_source.reset_index(drop=True), audit], axis=1)

assert len(manifest) == 600
assert manifest['video_id'].is_unique
assert manifest['label'].isin([0, 1]).all()
print('Technical validity by split and label:')
display(pd.crosstab([manifest['split'], manifest['label']], manifest['is_valid']))

Auditing MP4 files: 100%|██████████| 600/600 [16:23<00:00,  1.64s/it]

Technical validity by split and label:


is_valid          True
split      label      
train      0       240
           1       240
validation 0        60
           1        60

In [16]:
# Exact duplicate groups: same SHA-256 means byte-for-byte same MP4.
hash_counts = manifest['sha256'].dropna().value_counts()
duplicate_hashes = hash_counts[hash_counts > 1].index
exact_duplicate_groups = (
    manifest.loc[manifest['sha256'].isin(duplicate_hashes), ['video_id', 'video_path', 'label', 'split', 'sha256']]
    .sort_values(['sha256', 'video_id'])
    .reset_index(drop=True)
)
manifest['exact_duplicate_group'] = manifest['sha256'].where(manifest['sha256'].isin(duplicate_hashes))
exact_duplicate_groups.to_csv(EXACT_DUPLICATES_PATH, index=False)

print(f'Exact duplicate files: {len(exact_duplicate_groups)}')
display(exact_duplicate_groups.head(20))

Exact duplicate files: 0


,video_id,video_path,label,split,sha256


In [17]:
def hex_to_bits(value: str) -> np.ndarray:
    return np.unpackbits(np.frombuffer(bytes.fromhex(value), dtype=np.uint8))

hash_columns = [f'phash_{idx}' for idx in range(len(SAMPLE_RATIOS))]
hash_ready = manifest.dropna(subset=hash_columns).copy().reset_index(drop=True)
bits = np.stack([
    np.stack([hex_to_bits(row[column]) for column in hash_columns])
    for _, row in hash_ready.iterrows()
]) if len(hash_ready) else np.empty((0, len(SAMPLE_RATIOS), 64), dtype=np.uint8)

candidate_records = []
for left_idx in tqdm(range(max(len(hash_ready) - 1, 0)), desc='Comparing perceptual hashes'):
    # Align the same relative timestamps in both videos.
    hamming_per_sample = np.count_nonzero(bits[left_idx + 1:] != bits[left_idx], axis=2)
    mean_hamming = hamming_per_sample.mean(axis=1)
    for relative_right_idx in np.flatnonzero(mean_hamming <= NEAR_DUPLICATE_HAMMING_THRESHOLD):
        right_idx = left_idx + 1 + int(relative_right_idx)
        left = hash_ready.iloc[left_idx]
        right = hash_ready.iloc[right_idx]
        record = {
            'video_id_a': left.video_id, 'video_id_b': right.video_id,
            'label_a': left.label, 'label_b': right.label,
            'split_a': left.split, 'split_b': right.split,
            'cross_split': left.split != right.split,
            'mean_hamming': float(mean_hamming[relative_right_idx]),
        }
        record.update({
            f'hamming_{int(ratio * 100)}pct': int(hamming_per_sample[relative_right_idx, sample_idx])
            for sample_idx, ratio in enumerate(SAMPLE_RATIOS)
        })
        candidate_records.append(record)

near_duplicate_candidates = pd.DataFrame(candidate_records)
if near_duplicate_candidates.empty:
    near_duplicate_candidates = pd.DataFrame(columns=[
        'video_id_a', 'video_id_b', 'label_a', 'label_b', 'split_a', 'split_b', 'cross_split',
        *[f'hamming_{int(ratio * 100)}pct' for ratio in SAMPLE_RATIOS], 'mean_hamming'
    ])
else:
    near_duplicate_candidates = near_duplicate_candidates.sort_values('mean_hamming').reset_index(drop=True)

near_duplicate_candidates.to_csv(NEAR_DUPLICATES_PATH, index=False)
near_duplicate_ids = set(near_duplicate_candidates.get('video_id_a', pd.Series(dtype=str)).astype(str))
near_duplicate_ids.update(near_duplicate_candidates.get('video_id_b', pd.Series(dtype=str)).astype(str))
manifest['near_duplicate_candidate'] = manifest['video_id'].astype(str).isin(near_duplicate_ids)

print(f'Near-duplicate candidate pairs (manual review required): {len(near_duplicate_candidates)}')
print(f'Candidates crossing train/validation: {int(near_duplicate_candidates.cross_split.sum()) if len(near_duplicate_candidates) else 0}')
display(near_duplicate_candidates.head(20))

Comparing perceptual hashes: 100%|██████████| 599/599 [00:00<00:00, 3759.96it/s]

Near-duplicate candidate pairs (manual review required): 0
Candidates crossing train/validation: 0


,video_id_a,video_id_b,label_a,label_b,split_a,split_b,cross_split,hamming_10pct,hamming_30pct,hamming_50pct,hamming_70pct,hamming_90pct,mean_hamming


In [18]:
# Persist the central V2 source of truth only after all checks above complete.
required_manifest_columns = [
    'video_id', 'video_path', 'label', 'time_of_event', 'duration', 'fps',
    'frame_count', 'width', 'height', 'weather', 'light_conditions', 'scene',
    'split', 'is_valid', 'error_reason',
]
assert set(required_manifest_columns).issubset(manifest.columns)
assert manifest['split'].isin(['train', 'validation']).all()
assert not manifest.duplicated('video_id').any()
assert manifest.groupby('video_id')['split'].nunique().eq(1).all()

# Required columns first; audit diagnostics after them.
extra_columns = [column for column in manifest.columns if column not in required_manifest_columns]
manifest = manifest[required_manifest_columns + extra_columns].sort_values('video_id', key=lambda s: s.astype(int)).reset_index(drop=True)
manifest.to_csv(MANIFEST_PATH, index=False)

invalid = manifest.loc[~manifest['is_valid']].copy()
summary = {
    'dataset': 'nexar-ai/nexar_collision_prediction',
    'scope': 'selected_train_600',
    'total_videos': int(len(manifest)),
    'valid_videos': int(manifest['is_valid'].sum()),
    'invalid_videos': int((~manifest['is_valid']).sum()),
    'class_counts': {str(key): int(value) for key, value in manifest['label'].value_counts().sort_index().items()},
    'split_class_counts': {
        f'{split}|{label}': int(count)
        for (split, label), count in manifest.groupby(['split', 'label']).size().items()
    },
    'event_time_unit': 'seconds from video start',
    'event_time_range_seconds': [
        float(manifest.loc[manifest.label.eq(1), 'time_of_event'].min()),
        float(manifest.loc[manifest.label.eq(1), 'time_of_event'].max()),
    ],
    'exact_duplicate_files': int(len(exact_duplicate_groups)),
    'near_duplicate_candidate_pairs': int(len(near_duplicate_candidates)),
    'near_duplicate_cross_split_pairs': int(near_duplicate_candidates['cross_split'].sum()) if len(near_duplicate_candidates) else 0,
    'invalid_error_reasons': invalid['error_reason'].value_counts().to_dict(),
}
SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f'Manifest saved: {MANIFEST_PATH}')
print(f'Exact-duplicate report: {EXACT_DUPLICATES_PATH}')
print(f'Near-duplicate report: {NEAR_DUPLICATES_PATH}')
print(f'Audit summary: {SUMMARY_PATH}')
display(pd.DataFrame([summary]))
display(invalid[required_manifest_columns + ['file_size_bytes']].head(20))

Manifest saved: P:\NexarCollisionData\video_manifest_v2.csv
Exact-duplicate report: P:\NexarCollisionData\exact_duplicate_groups_v2.csv
Near-duplicate report: P:\NexarCollisionData\near_duplicate_candidates_v2.csv
Audit summary: P:\NexarCollisionData\data_audit_summary_v2.json


,dataset,scope,total_videos,valid_videos,invalid_videos,class_counts,split_class_counts,event_time_unit,event_time_range_seconds,exact_duplicate_files,near_duplicate_candidate_pairs,near_duplicate_cross_split_pairs,invalid_error_reasons
0,nexar-ai/nexar_collision_prediction,selected_train_600,600,600,0,"{'0': 300, '1': 300}","{'train|0': 240, 'train|1': 240, 'validation|0...",seconds from video start,"[3.921, 56.8]",0,0,0,{}


,video_id,video_path,label,time_of_event,duration,fps,frame_count,width,height,weather,light_conditions,scene,split,is_valid,error_reason,file_size_bytes


## شرط پایان مرحلهٔ ۱

قبل از رفتن به مرحلهٔ ۲ باید این‌ها را بررسی کنیم: تعداد valid/invalid، علت هر نامعتبر، هر duplicate دقیق و هر candidate مشابهی که بین `train` و `validation` افتاده است. فقط بعد از تأیید این گزارش، split V2 را به‌صورت نهایی freeze می‌کنیم و `sequence_manifest_v2.csv` را می‌سازیم.